<a href="https://colab.research.google.com/github/moazmohemeda-boop/Autism-cnn/blob/main/CNN_ens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications import MobileNet, Xception, EfficientNetB0, ResNet50V2
from tensorflow.keras.applications.mobilenet import preprocess_input as mob_pre
from tensorflow.keras.applications.xception import preprocess_input as xc_pre
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet_v2 import preprocess_input as res_pre

from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.losses import BinaryCrossentropy

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.optimize import differential_evolution

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE
BATCH_224 = 32
BATCH_299 = 16

label_map = {
    "autistic": 1,
    "non_autistic": 0
}

base_dir = "/kaggle/input/datasets/prayashdas/autistic-children-facial-image-dataset"

print("Ready")

Ready


In [ ]:
# دي بتتغير علي حسب انتو فاتحين كاجل ولا فيجوال
def collect_images(folder):
    data = []

    for label in ["autistic", "non_autistic"]:
        class_dir = os.path.join(folder, label)

        for file in sorted(os.listdir(class_dir)):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                data.append({
                    "filepath": os.path.join(class_dir, file),
                    "label": label
                })

    return pd.DataFrame(data)

df_train = collect_images(os.path.join(base_dir, "train"))
df_val = collect_images(os.path.join(base_dir, "valid"))
df_test = collect_images(os.path.join(base_dir, "test"))

y_train = np.array([label_map[x] for x in df_train["label"]], dtype=np.float32)
y_val = np.array([label_map[x] for x in df_val["label"]], dtype=np.float32)
y_test = np.array([label_map[x] for x in df_test["label"]], dtype=np.float32)

print("Train:", len(df_train))
print("Validation:", len(df_val))
print("Test:", len(df_test))

print("\nTrain:")
print(df_train["label"].value_counts())

print("\nValidation:")
print(df_val["label"].value_counts())

print("\nTest:")
print(df_test["label"].value_counts())

Train: 2526
Validation: 200
Test: 200

Train:
label
autistic        1263
non_autistic    1263
Name: count, dtype: int64

Validation:
label
autistic        100
non_autistic    100
Name: count, dtype: int64

Test:
label
autistic        100
non_autistic    100
Name: count, dtype: int64


In [ ]:

def load_raw(path, label, size):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [size, size])
    return tf.cast(img, tf.float32), tf.cast(label, tf.float32)


def augment(image, label):
    image = tf.image.random_flip_left_right(image)

    image = tf.image.random_brightness(
        image,
        max_delta=12.0
    )

    image = tf.image.random_contrast(
        image,
        lower=0.90,
        upper=1.10
    )

    image = tf.clip_by_value(image, 0.0, 255.0)

    return image, label


def make_dataset(df, size, preprocess_fn, batch_size, training=False):

    paths = df["filepath"].values
    labels = np.array(
        [label_map[x] for x in df["label"]],
        dtype=np.float32
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda p, y: load_raw(p, y, size),
        num_parallel_calls=AUTOTUNE
    )

    if training:
        ds = ds.map(
            augment,
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.map(
        lambda x, y: (preprocess_fn(x), y),
        num_parallel_calls=AUTOTUNE
    )

    return ds.batch(batch_size).prefetch(AUTOTUNE)


ds_tr_mob = make_dataset(
    df_train, 224, mob_pre, BATCH_224, True
)

ds_va_mob = make_dataset(
    df_val, 224, mob_pre, BATCH_224
)

ds_te_mob = make_dataset(
    df_test, 224, mob_pre, BATCH_224
)


ds_tr_xc = make_dataset(
    df_train, 299, xc_pre, BATCH_299, True
)

ds_va_xc = make_dataset(
    df_val, 299, xc_pre, BATCH_299
)

ds_te_xc = make_dataset(
    df_test, 299, xc_pre, BATCH_299
)


ds_tr_eff = make_dataset(
    df_train, 224, eff_pre, BATCH_224, True
)

ds_va_eff = make_dataset(
    df_val, 224, eff_pre, BATCH_224
)

ds_te_eff = make_dataset(
    df_test, 224, eff_pre, BATCH_224
)


ds_tr_res = make_dataset(
    df_train, 224, res_pre, BATCH_224, True
)

ds_va_res = make_dataset(
    df_val, 224, res_pre, BATCH_224
)

ds_te_res = make_dataset(
    df_test, 224, res_pre, BATCH_224
)

print("Datasets ready")

I0000 00:00:1788366896.880518    2193 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Datasets ready


In [ ]:

def callbacks(name):
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=7,
            mode="min",
            restore_best_weights=True
        ),
        ModelCheckpoint(
            f"{name}.keras",
            monitor="val_accuracy",
            mode="max",
            save_best_only=True
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.25,
            patience=2,
            min_lr=1e-7,
            mode="min",
            verbose=1
        )
    ]


def build_model(base, dropout=0.4, dense_units=256):

    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)
    x = Dense(
        dense_units,
        activation="relu"
    )(x)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)

    output = Dense(
        1,
        activation="sigmoid"
    )(x)

    return Model(
        base.input,
        output
    )


def fine_tune(base, last_layers):

    base.trainable = True

    for layer in base.layers[:-last_layers]:
        layer.trainable = False

    for layer in base.layers:
        if isinstance(layer, BatchNormalization):
            layer.trainable = False

In [ ]:

print("MobileNet")

base = MobileNet(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

model = build_model(base, 0.35, 256)

model.compile(
    optimizer=Adam(1e-3),
    loss=BinaryCrossentropy(label_smoothing=0.02),
    metrics=["accuracy"]
)

model.fit(
    ds_tr_mob,
    validation_data=ds_va_mob,
    epochs=20,
    callbacks=callbacks("best_mobilenet")
)

fine_tune(base, 45)

model.compile(
    optimizer=Adam(2e-5),
    loss=BinaryCrossentropy(label_smoothing=0.01),
    metrics=["accuracy"]
)

model.fit(
    ds_tr_mob,
    validation_data=ds_va_mob,
    epochs=30,
    callbacks=callbacks("best_mobilenet")
)

del model, base
tf.keras.backend.clear_session()
gc.collect()


print("Xception")

base = Xception(
    input_shape=(299, 299, 3),
    include_top=False,
    weights="imagenet"
)

model = build_model(base, 0.35, 256)

model.compile(
    optimizer=Adam(1e-3),
    loss=BinaryCrossentropy(label_smoothing=0.02),
    metrics=["accuracy"]
)

model.fit(
    ds_tr_xc,
    validation_data=ds_va_xc,
    epochs=20,
    callbacks=callbacks("best_xception")
)

fine_tune(base, 70)

model.compile(
    optimizer=Adam(1e-5),
    loss=BinaryCrossentropy(label_smoothing=0.01),
    metrics=["accuracy"]
)

model.fit(
    ds_tr_xc,
    validation_data=ds_va_xc,
    epochs=30,
    callbacks=callbacks("best_xception")
)

del model, base
tf.keras.backend.clear_session()
gc.collect()

MobileNet
17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20


I0000 00:00:1788366909.781130    2303 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


79/79 ━━━━━━━━━━━━━━━━━━━━ 27s 220ms/step - accuracy: 0.7253 - loss: 0.6505 - val_accuracy: 0.6850 - val_loss: 0.6527 - learning_rate: 0.0010
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.8096 - loss: 0.4602 - val_accuracy: 0.7650 - val_loss: 0.5081 - learning_rate: 0.0010
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.8325 - loss: 0.3966 - val_accuracy: 0.7950 - val_loss: 0.4851 - learning_rate: 0.0010
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.8511 - loss: 0.3704 - val_accuracy: 0.7950 - val_loss: 0.4595 - learning_rate: 0.0010
Epoch 5/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.8531 - loss: 0.3543 - val_accuracy: 0.7900 - val_loss: 0.4870 - learning_rate: 0.0010
Epoch 6/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.8638 - loss: 0.3247 - val_accuracy: 0.8450 - val_loss: 0.4373 - learning_rate: 0.0010
Epoch 7/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.8832 - loss: 0.3056 - val_accuracy: 0.

2026-09-02 16:36:16.463374: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:36:16.665361: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8736 - loss: 0.3019

2026-09-02 16:36:24.921885: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:36:25.124293: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


... (3 outputs left out)

2026-09-02 16:42:21.198142: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:42:21.386961: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:42:22.298738: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:42:22.498540: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:42:22.903298: E external/local_xla/xla/stream_

157/158 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.9018 - loss: 0.2784

2026-09-02 16:43:04.212900: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:43:04.401648: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:43:05.265402: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:43:05.464600: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-02 16:43:05.858328: E external/local_xla/xla/stream_

158/158 ━━━━━━━━━━━━━━━━━━━━ 100s 425ms/step - accuracy: 0.8911 - loss: 0.2857 - val_accuracy: 0.7950 - val_loss: 0.4895 - learning_rate: 1.0000e-05
Epoch 2/30
158/158 ━━━━━━━━━━━━━━━━━━━━ 22s 140ms/step - accuracy: 0.9101 - loss: 0.2478 - val_accuracy: 0.8050 - val_loss: 0.4648 - learning_rate: 1.0000e-05
Epoch 3/30
158/158 ━━━━━━━━━━━━━━━━━━━━ 22s 140ms/step - accuracy: 0.9220 - loss: 0.2220 - val_accuracy: 0.8200 - val_loss: 0.4793 - learning_rate: 1.0000e-05
Epoch 4/30
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.9363 - loss: 0.1940
Epoch 4: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-06.
158/158 ━━━━━━━━━━━━━━━━━━━━ 21s 132ms/step - accuracy: 0.9402 - loss: 0.1923 - val_accuracy: 0.8050 - val_loss: 0.4837 - learning_rate: 1.0000e-05
Epoch 5/30
158/158 ━━━━━━━━━━━━━━━━━━━━ 21s 133ms/step - accuracy: 0.9446 - loss: 0.1780 - val_accuracy: 0.8200 - val_loss: 0.4696 - learning_rate: 2.5000e-06
Epoch 6/30
158/158 ━━━━━━━━━━━━━━━━━━━━ 21s 133ms/step - accur

In [ ]:

print("EfficientNetB0")

base = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

model = build_model(base, 0.35, 256)

model.compile(
    optimizer=Adam(1e-3),
    loss=BinaryCrossentropy(label_smoothing=0.02),
    metrics=["accuracy"]
)

model.fit(
    ds_tr_eff,
    validation_data=ds_va_eff,
    epochs=20,
    callbacks=callbacks("best_eff")
)

fine_tune(base, 60)

model.compile(
    optimizer=Adam(1e-5),
    loss=BinaryCrossentropy(label_smoothing=0.01),
    metrics=["accuracy"]
)

model.fit(
    ds_tr_eff,
    validation_data=ds_va_eff,
    epochs=30,
    callbacks=callbacks("best_eff")
)

del model, base
tf.keras.backend.clear_session()
gc.collect()


print("ResNet50V2")

base = ResNet50V2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

model = build_model(base, 0.35, 256)

model.compile(
    optimizer=Adam(1e-3),
    loss=BinaryCrossentropy(label_smoothing=0.02),
    metrics=["accuracy"]
)

model.fit(
    ds_tr_res,
    validation_data=ds_va_res,
    epochs=20,
    callbacks=callbacks("best_resnet")
)

fine_tune(base, 50)

model.compile(
    optimizer=Adam(1e-5),
    loss=BinaryCrossentropy(label_smoothing=0.01),
    metrics=["accuracy"]
)

model.fit(
    ds_tr_res,
    validation_data=ds_va_res,
    epochs=30,
    callbacks=callbacks("best_resnet")
)

del model, base
tf.keras.backend.clear_session()
gc.collect()

79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 72ms/step - accuracy: 0.9929 - loss: 0.0788 - val_accuracy: 0.8700 - val_loss: 0.3725 - learning_rate: 1.0000e-07
Epoch 20/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.9865 - loss: 0.0859 - val_accuracy: 0.8700 - val_loss: 0.3719 - learning_rate: 1.0000e-07
Epoch 21/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.9917 - loss: 0.0770 - val_accuracy: 0.8700 - val_loss: 0.3717 - learning_rate: 1.0000e-07
Epoch 22/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.9909 - loss: 0.0783 - val_accuracy: 0.8700 - val_loss: 0.3734 - learning_rate: 1.0000e-07
Epoch 23/30
79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 70ms/step - accuracy: 0.9925 - loss: 0.0777 - val_accuracy: 0.8700 - val_loss: 0.3725 - learning_rate: 1.0000e-07


0

In [ ]:

best_mob = load_model("best_mobilenet.keras")
best_xc = load_model("best_xception.keras")
best_eff = load_model("best_eff.keras")
best_res = load_model("best_resnet.keras")


def predict_tta(model, ds):

    p1 = model.predict(
        ds,
        verbose=0
    ).flatten()

    p2 = []

    for x, _ in ds:
        xf = tf.image.flip_left_right(x)

        pred = model(
            xf,
            training=False
        ).numpy().flatten()

        p2.extend(pred)

    p2 = np.array(p2)

    return (p1 + p2) / 2.0


val_mob = predict_tta(best_mob, ds_va_mob)
val_xc = predict_tta(best_xc, ds_va_xc)
val_eff = predict_tta(best_eff, ds_va_eff)
val_res = predict_tta(best_res, ds_va_res)


test_mob = predict_tta(best_mob, ds_te_mob)
test_xc = predict_tta(best_xc, ds_te_xc)
test_eff = predict_tta(best_eff, ds_te_eff)
test_res = predict_tta(best_res, ds_te_res)


val_matrix = np.vstack([
    val_mob,
    val_xc,
    val_eff,
    val_res
]).T

test_matrix = np.vstack([
    test_mob,
    test_xc,
    test_eff,
    test_res
]).T


def objective(params):

    weights = np.array(params[:4])
    weights = weights / np.sum(weights)

    threshold = params[4]

    probs = np.sum(
        val_matrix * weights,
        axis=1
    )

    preds = (
        probs >= threshold
    ).astype(int)

    return -accuracy_score(
        y_val,
        preds
    )


bounds = [
    (0.05, 0.70),
    (0.05, 0.70),
    (0.05, 0.70),
    (0.05, 0.70),
    (0.35, 0.65)
]


result = differential_evolution(
    objective,
    bounds,
    seed=SEED,
    maxiter=80,
    popsize=12,
    polish=True,
    workers=1
)


params = result.x

weights = np.array(params[:4])
weights = weights / np.sum(weights)

threshold = params[4]

val_probs = np.sum(
    val_matrix * weights,
    axis=1
)

val_preds = (
    val_probs >= threshold
).astype(int)

print(
    "Validation Accuracy:",
    accuracy_score(
        y_val,
        val_preds
    ) * 100
)

print(
    "Weights:",
    weights
)

print(
    "Threshold:",
    threshold
)

Validation Accuracy: 91.0
Weights: [0.19519088 0.2809816  0.18689833 0.33692919]
Threshold: 0.5147751683843053


In [ ]:

test_probs = np.sum(
    test_matrix * weights,
    axis=1
)

test_preds = (
    test_probs >= threshold
).astype(int)

test_accuracy = accuracy_score(
    y_test,
    test_preds
) * 100

print("\n" + "=" * 70)
print(
    f"UNSEEN TEST ACCURACY: {test_accuracy:.2f}%"
)
print("=" * 70)

print(
    "\nClassification Report:\n"
)

print(
    classification_report(
        y_test,
        test_preds,
        target_names=[
            "non_autistic",
            "autistic"
        ],
        digits=4
    )
)

print(
    "\nConfusion Matrix:\n"
)

print(
    confusion_matrix(
        y_test,
        test_preds
    )
)

print("\nFinal Weights:")
print(
    f"MobileNet : {weights[0]:.4f}"
)
print(
    f"Xception  : {weights[1]:.4f}"
)
print(
    f"Efficient : {weights[2]:.4f}"
)
print(
    f"ResNet    : {weights[3]:.4f}"
)

print(
    f"\nThreshold: {threshold:.4f}"
)


UNSEEN TEST ACCURACY: 91.00%

Classification Report:

              precision    recall  f1-score   support

non_autistic     0.8868    0.9400    0.9126       100
    autistic     0.9362    0.8800    0.9072       100

    accuracy                         0.9100       200
   macro avg     0.9115    0.9100    0.9099       200
weighted avg     0.9115    0.9100    0.9099       200


Confusion Matrix:

[[94  6]
 [12 88]]

Final Weights:
MobileNet : 0.1952
Xception  : 0.2810
Efficient : 0.1869
ResNet    : 0.3369

Threshold: 0.5148
